### Exploration of YAML and RAG Classifier Module 

```
# Classification Module

This module provides two main classifiers for text classification tasks: `YamlClassifier` and `RAGClassifier`. Both classifiers are designed to categorize user queries into predefined labels based on a set of examples and descriptions.

## YamlClassifier

The `YamlClassifier` is a simple yet powerful classifier that uses a YAML configuration file to define the classification task, labels, and examples.

### Features

- Load classification configuration from a YAML file
- Support for multiple labels with positive and negative examples
- Customizable number of examples to use for each label
- Synchronous and asynchronous prediction methods

## RAGClassifier

> Before running the `RAGClassifier` script, make sure to set an OPENAI_API_KEY variable in your shell.

The `RAGClassifier` is a classifier that uses a retreival model to classify user queries into predefined labels based on a set of examples and descriptions.

### Features

- Inherits all features from YamlClassifier
- Uses a vector database (ChromaDB) to store and retrieve similar examples
- Dynamically fetches similar examples for each query during classification
- Customizable number of similar examples to fetch (default: 2)
- Provides distance metrics for retrieved examples to aid in classification
- Supports fitting the classifier with examples and loading pre-fitted databases
```

In [4]:
# %pip install chromadb jinja2

In [4]:
from rich import inspect as rinspect
from rich import print as rprint

from fastcore.all import *

#### YAML Classifier

In [3]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Dict, Any, Optional, TypeVar, Type
from textwrap import dedent
from instructor import Instructor
from jinja2 import Template
import openai, instructor

T = TypeVar('T' , bound='BaseModel')

In [2]:
## rinspect(field_validator, help=True)

In [5]:
class Example(BaseModel):
    positive: List[str]
    negative: List[str]

class Label(BaseModel):
    name: str
    description: str
    examples: Example

    @field_validator('name')
    def name_must_be_snake_case(cls, v):
        import re
        if not re.match(r'^[a-z0-9_]+$', v): raise ValueError('must be a snake_case string')
        return v

In [6]:
example = Example(positive=['a', 'b'], negative=['c', 'd'])
expected = {'positive': ['a', 'b'], 'negative': ['c', 'd']}
test_eq(example.model_dump(mode='json'), expected)

In [44]:
import yaml
fname = 'copy_example.yaml'
with open(fname, 'r') as f: data = yaml.safe_load(f)
data

{'task': 'query_type_classification',
 'description': 'Classify user queries into predefined categories based on the type of task requested.  This classification system aims to categorize incoming user queries into specific task types,  such as authorization requests, content drafting, or time-sensitive information retrieval. \nThe purpose is to streamline query processing, improve response accuracy, and potentially  route queries to appropriate handling systems or personnel. This classification helps in  prioritizing tasks, applying relevant security measures, and ensuring that time-sensitive  queries are handled promptly. \nThe system uses a set of predefined labels, each with a clear description and examples,  to accurately categorize a wide range of user inputs. This approach enables efficient  query handling and enhances the overall user experience by providing more targeted and  timely responses to various types of requests.\n',
 'n_examples': 2,
 'n_similar_examples': 2,
 'label

In [9]:
expected = ['task', 'description', 'n_examples', 'n_similar_examples', 'labels']
test_eq(list(data.keys()), expected)

expected = ['authorization','drafting','time_filter_requirement']
test_eq(L(data['labels']).attrgot('name'), expected)

In [10]:
expected = '<task> hello </task> <description> world </description>'
tmpl = Template(dedent('''<task> {{ task }} </task> <description> {{ description }} </description>'''))
test_eq(tmpl.render(task='hello', description='world'), expected) 

o = dict(task='hello', description='world')
test_eq(tmpl.render(**o), expected)

In [11]:
def get_template():
    template = Template(
                dedent(
                    """
            <task>
                {{ task }}
            </task>

            <description>
                {{ description }}
            </description>

            <labels>
            {% for label in labels %}
                <label>
                    <name>
                        {{ label.name }}
                    </name>

                    <description>
                        {{ label.description }}
                    </description>

                    <examples>
                        <positive>
                        {% for example in label.examples.positive[:n_examples] %}
                            <example>
                                {{ example }}
                            </example>
                        {% endfor %}
                        </positive>

                        <negative>
                        {% for example in label.examples.negative[:n_examples] %}
                            <example>
                                {{ example }}
                            </example>
                        {% endfor %}
                        </negative>
                    </examples>
                </label>
            {% endfor %}
            </labels>

            Instructions:
            1. Carefully read the user's query.
            2. Compare the query to the descriptions and examples for each label.
            3. Use the provided examples as a guide:
            - Positive examples show queries that should be classified under that label.
            - Negative examples show queries that should not be classified under that label.
            4. Consider both the content and the intent of the query when matching to a label.
            5. Choose the most appropriate label that matches the query's intent and content.
            6. If the query doesn't clearly fit any label, choose the closest match based on similarity to the examples and description.
            7. Provide your classification as a single word matching the chosen label's name.
            8. Do not assume any specific task unless it's explicitly mentioned in the 'task' variable.
            """
                )
            )
    #rprint(template.render(task=data['task'], description=data['description'], labels=data['labels'], n_examples=data['n_examples']))
    return template

## YamlClassifier

In [47]:
class YamlClassifier(BaseModel):
    task: str
    description: str
    n_examples: int | None = Field(
        default=100, description='Number of examples to use for each label'
    )
    # n_similar_examples: int
    labels: List[Label]

    @classmethod
    def load(cls, fname: str):
        import yaml
        with open(fname, 'r') as f: data = yaml.safe_load(f)
        return cls(**data)

    def to_system_messages(self) -> str: return get_template().render(**self.model_dump())

    def get_user_query(self, query: str) -> str:
        return f'Correctly Classify:\n\n{query}'
    
    def get_labels(self) -> List[str]: return L(self.labels).attrgot('name')
    
    def predict(
        self, query: str, model: str, response_model: Type[T], client: Instructor
    ):
        user_query = self.get_user_query(query)
        system_messages = self.to_system_messages()
        return client.create(
            model=model, 
            response_model=response_model,
            messages=[
                {'role': 'system', 'content': system_messages},
                {'role': 'user', 'content': user_query}
            ],
        )

In [48]:
classifier = YamlClassifier.load(fname)

class Prediction(BaseModel):
    correct_labels: List[str] = Field(
        description='The predicted label(s) as a list of strings'
    )

class PredictionWithReasoning(Prediction):
    reasoning: str = Field(
        description="A detailed explanation of the thought process leading to the prediction, including key factors considered, comparisons to label descriptions and examples, and how the query's content and intent align with the chosen label"
    )

In [14]:
client = instructor.from_openai(openai.OpenAI())
resp = classifier.predict(
    query="When was the last time I ask you about dinner?",
    response_model=Prediction,
    client=client,
    model='gpt-4o-mini'
)

In [15]:
rprint(resp.model_dump_json(indent=2))

{
  "correct_labels": [
    "time_filter_requirement"
  ]
}

In [25]:
def segment_user_query(client, query: str, reasoning: bool = False, model: str = 'gpt-4o-mini'):
    response_model = PredictionWithReasoning if reasoning else Prediction
    resp = classifier.predict(query=query, response_model=response_model, client=client, model=model)
    return resp

In [37]:
client = instructor.from_openai(openai.OpenAI())
res = segment_user_query(client, query="When was the last time I ask you about dinner?", reasoning=False)
rprint(res)

Prediction(correct_labels=['time_filter_requirement'])

In [36]:
test_eq(res.correct_labels, ['time_filter_requirement'])
test_eq(res.model_dump_json(), '''{"correct_labels":["time_filter_requirement"]}''')

In [40]:
client = instructor.from_openai(openai.OpenAI())
res = segment_user_query(client, query="When was the last time I ask you about dinner?", reasoning=True)
rprint(res)

PredictionWithReasoning(
    correct_labels=['time_filter_requirement'],
    reasoning="The query references a specific timing ('last time') related to a previous inquiry about dinner. It 
reflects a need for information that is time-sensitive in nature, as it asks for a past occurrence that is relevant
to the present context of planning or discussing dinner."
)

In [41]:
test_eq(res.correct_labels, ['time_filter_requirement'])
test_eq(res.reasoning is not None, True)

In [51]:
query='Leapp preupgrade error:- Actor: target_userspace_creator Message: Unable to install RHEL 8 userspace packages.'
res = segment_user_query(client, query=query, reasoning=True)

In [52]:
rprint(res)

PredictionWithReasoning(
    correct_labels=['error_symptoms'],
    reasoning="The query presented is directly related to an error encountered during a process (Leapp preupgrade) 
where the user is unable to install certain packages. This falls under the label 'error_symptoms' as it describes a
specific error message that the user is facing. The description for the 'error_symptoms' label matches with the 
intent and content of the query, while other labels do not apply as there is no request for authorization, drafting
content, or a time-sensitive query present."
)

In [53]:
test_eq(res.correct_labels, ['error_symptoms'])

In [54]:
query='failed to upgrade with leapp Stderr: Failed to create directory /var/lib/leapp/el8userspace//sys/fs/selinux: Read-only file system'
rprint(segment_user_query(client, query=query, reasoning=True))

PredictionWithReasoning(
    correct_labels=['error_symptoms'],
    reasoning="The user's query indicates a specific error encountered during an upgrade process involving the 
leapp tool. The phrase 'failed to upgrade' and the details about the directory creation failure clearly align with 
the label 'error_symptoms,' which categorizes queries that relate to resolving errors faced. The query provides a 
concrete error message ('Read-only file system') that supports the classification."
)

In [55]:
test_eq(res.correct_labels, ['error_symptoms'])

## RAGClassifier

In [57]:
import chromadb
from jinja2 import Template
from yaml_classifier import YamlClassifier
from textwrap import dedent
from pydantic import Field

coll_name = 'example'

In [58]:
cli = chromadb.Client()

In [59]:
rinspect(cli, help=True)

╭─────────────────────── <class 'chromadb.api.client.Client'> ───────────────────────╮
│ A client for Chroma. This is the main entrypoint for interacting with Chroma.      │
│ A client internally stores its tenant and database and proxies calls to a          │
│ Server API instance of Chroma. It treats the Server API and corresponding System   │
│ as a singleton, so multiple clients connecting to the same resource will share the │
│ same API instance.                                                                 │
│                                                                                    │
│ Client implementations should be implement their own API-caching strategies.       │
│                                                                                    │
│ ╭────────────────────────────────────────────────────────────────────────────────╮ │
│ │ <chromadb.api.client.Client object at 0x7f340c435990>                          │ │
│ ╰────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                    │
│ database = 'default_database'                                                      │
│   tenant = 'default_tenant'                                                        │
╰────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
rinspect(cli.get_or_create_collection, help=True)

╭──── <bound method Client.get_or_create_collection of <chromadb.api.client.Client object at 0x7fe7b7fdb2b0>> ────╮
│ def Client.get_or_create_collection(name: str, configuration:                                                   │
│ Optional[chromadb.api.configuration.CollectionConfigurationInterface] = None, metadata: Optional[Dict[str,      │
│ Any]] = None, embedding_function: Optional[chromadb.api.types.EmbeddingFunction[Union[List[str],                │
│ List[numpy.ndarray[Any, numpy.dtype[Union[numpy.uint64, numpy.int64, numpy.float64]]]]]]] =                     │
│ <chromadb.utils.embedding_functions.onnx_mini_lm_l6_v2.ONNXMiniLM_L6_V2 object at 0x7fe7d82f0280>, data_loader: │
│ Optional[chromadb.api.types.DataLoader[List[Optional[numpy.ndarray[Any, numpy.dtype[Union[numpy.uint64,         │
│ numpy.int64, numpy.float64]]]]]]] = None) -> chromadb.api.models.Collection.Collection:                         │
│                                                                                                                 │
│ Get or create a collection with the given name and metadata.                                                    │
│ Args:                                                                                                           │
│     name: The name of the collection to get or create                                                           │
│     metadata: Optional metadata to associate with the collection. If                                            │
│     the collection alredy exists, the metadata will be updated if                                               │
│     provided and not None. If the collection does not exist, the                                                │
│     new collection will be created with the provided metadata.                                                  │
│     embedding_function: Optional function to use to embed documents                                             │
│     data_loader: Optional function to use to load records (documents, images, etc.)                             │
│                                                                                                                 │
│ Returns:                                                                                                        │
│     The collection                                                                                              │
│                                                                                                                 │
│ Examples:                                                                                                       │
│     ```python                                                                                                   │
│     client.get_or_create_collection("my_collection")                                                            │
│     # collection(name="my_collection", metadata={})                                                             │
│     ```                                                                                                         │
│                                                                                                                 │
│ 28 attribute(s) not shown. Run inspect(inspect) for options.                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [94]:
import chromadb.utils.embedding_functions as embedding_functions

In [95]:
rinspect(embedding_functions, help=True)

╭─ <module 'chromadb.utils.embedding_functions' from '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-p─╮
│       amazon_bedrock_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.amazon_bedrock_embedding_functio… │
│                                           from                                                                  │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│     chroma_langchain_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.chroma_langchain_embedding_funct… │
│                                           from                                                                  │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│               cohere_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.cohere_embedding_function' from   │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│               google_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.google_embedding_function' from   │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│          huggingface_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.huggingface_embedding_function'   │
│                                           from                                                                  │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│                               importlib = <module 'importlib' from                                              │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/importlib/__in… │
│           instructor_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.instructor_embedding_function'    │
│                                           from                                                                  │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│                          is_thin_client = False                                                                 │
│                 jina_embedding_function = <module 'chromadb.utils.embedding_functions.jina_embedding_function'  │
│                                           from                                                                  │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│               ollama_embedding_function = <module                                                               │
│                                           'chromadb.utils.embedding_functions.ollama_embedding_function' from   │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│                      onnx_mini_lm_l6_v2 = <module 'chromadb.utils.embedding_functions.onnx_mini_lm_l6_v2' from  │
│                                           '/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/… │
│            open_clip_embedding_function = <module     

In [96]:
db = cli.get_or_create_collection(coll_name, 
                                  embedding_function=embedding_functions.OpenAIEmbeddingFunction(
                                      api_key=os.getenv('OPENAI_API_KEY'),
                                      model_name='text-embedding-3-small'
                                  ))

In [97]:
db

Collection(id=1e72f6d3-c6ae-4f7e-a5f3-12e60deb982d, name=example)

#### Using Default Embedding Function

Before change `db._embedding_function` returns
<chromadb.utils.embedding_functions.onnx_mini_lm_l6_v2.ONNXMiniLM_L6_V2 at 0x7f340dd79780>

If no embedding function is provided, MiniLM L6 is used.

In [99]:
# After change 
db._embedding_function

In [100]:
test_eq(len(classifier.labels), 4)

In [101]:
classifier.labels

[Label(name='authorization', description='Queries related to granting or verifying access permissions', examples=Example(positive=['Can you give me access to the financial reports?', 'I need permission to edit the shared document.'], negative=["What's the weather like today?", 'Can you draft an email for me?'])),
 Label(name='drafting', description='Queries requesting the creation or composition of content', examples=Example(positive=['Write a summary of the quarterly report.', 'Can you draft a reply to this customer complaint?'], negative=["What's the capital of France?", 'Do I have permission to access the server room?'])),
 Label(name='time_filter_requirement', description='Queries that may need to be filtered based on time-sensitive information', examples=Example(positive=['What are the current stock prices for Tech Co.?', "Give me today's headlines from major news outlets."], negative=['Who was the first president of the United States?', "What's the chemical formula for water?"]))

In [102]:
classifier.get_labels()

(#4) ['authorization','drafting','time_filter_requirement','error_symptoms']

In [103]:
all_egs = []
for label in classifier.labels:
    for each in label.examples.positive:
        all_egs.append(dict(text=each, label=label.name))
    for each in label.examples.negative:
        all_egs.append(dict(text=each, label=label.name))


In [104]:
all_egs

[{'text': 'Can you give me access to the financial reports?',
  'label': 'authorization'},
 {'text': 'I need permission to edit the shared document.',
  'label': 'authorization'},
 {'text': "What's the weather like today?", 'label': 'authorization'},
 {'text': 'Can you draft an email for me?', 'label': 'authorization'},
 {'text': 'Write a summary of the quarterly report.', 'label': 'drafting'},
 {'text': 'Can you draft a reply to this customer complaint?',
  'label': 'drafting'},
 {'text': "What's the capital of France?", 'label': 'drafting'},
 {'text': 'Do I have permission to access the server room?',
  'label': 'drafting'},
 {'text': 'What are the current stock prices for Tech Co.?',
  'label': 'time_filter_requirement'},
 {'text': "Give me today's headlines from major news outlets.",
  'label': 'time_filter_requirement'},
 {'text': 'Who was the first president of the United States?',
  'label': 'time_filter_requirement'},
 {'text': "What's the chemical formula for water?",
  'label

In [105]:
db.upsert(
    documents=L(all_egs).attrgot('text'),
    metadatas=[dict(label=l['label']) for l in all_egs],
    ids=[str(i) for i, _ in enumerate(all_egs)]
)

InvalidDimensionException: Embedding dimension 1536 does not match collection dimensionality 384

From [perplexity](https://www.perplexity.ai/search/chroma-if-i-create-a-collectio-XQDWcUEKTDi9SP.gU12e_A)

When working with Chroma collections and embedding functions, you're correct that changing the embedding function for an existing collection is not straightforward. Here's a detailed explanation:

#### Embedding Functions and Collections

1. **Embedding Function Binding**: When you create a collection in Chroma, the embedding function is tightly coupled with that collection[1]. This means the embedding function you specify at creation time becomes an integral part of how that collection operates.

2. **Consistency Requirement**: Chroma requires consistency in the embedding function used for a collection. This is because the embeddings generated by different functions may have different dimensions or live in different vector spaces, making them incompatible[2].

#### Changing Embedding Functions

If you want to switch to a new embedding function for your data, you generally need to follow these steps:

1. **Create a New Collection**: You'll need to create a new collection with the desired embedding function[1].

2. **Re-embed and Add Data**: You must re-embed all your documents using the new embedding function and add them to the new collection[1].

3. **Delete Old Collection**: Once you've migrated your data to the new collection, you can delete the old collection if it's no longer needed[3].

#### Why Deletion is Necessary

The reason for creating a new collection rather than modifying an existing one is:

1. **Vector Incompatibility**: Embeddings from different functions may have different dimensions or semantic meanings, making them incompatible within the same collection[2].

2. **Consistency in Querying**: To ensure accurate similarity searches, all embeddings in a collection must be generated by the same function[2].

3. **Data Integrity**: Mixing embeddings from different functions could lead to unreliable results in similarity searches and other operations[2].

#### Best Practices

- **Plan Ahead**: Choose your embedding function carefully at the outset to minimize the need for changes.
- **Version Your Collections**: When making significant changes like switching embedding functions, create new collections with version numbers or descriptive names.
- **Backup Data**: Always ensure you have your original documents backed up, so you can re-embed them if needed.

In conclusion, while it may seem inconvenient, creating a new collection with the desired embedding function and re-embedding your data is the most reliable way to switch embedding functions in Chroma. This approach ensures data consistency and maintains the integrity of your vector search capabilities.

Citations:
[1] https://www.datacamp.com/tutorial/chromadb-tutorial-step-by-step-guide
[2] https://www.reddit.com/r/LangChain/comments/1er9221/chroma_embedding_function/
[3] https://github.com/langchain-ai/langchain/issues/4519
[4] https://docs.trychroma.com/guides
[5] https://blog.gopenai.com/efficient-document-embedding-management-with-chromadb-deleting-resetting-and-more-dac0e70e713b
[6] https://github.com/langchain-ai/langchain/discussions/17797
[7] https://cookbook.chromadb.dev/faq/
[8] https://stackoverflow.com/questions/76379440/how-to-see-the-embedding-of-the-documents-with-chroma-or-any-other-db-saved-in

In [110]:
cli.delete_collection(coll_name)

In [111]:
db = cli.get_or_create_collection(coll_name, 
                                  embedding_function=embedding_functions.OpenAIEmbeddingFunction(
                                      api_key=os.getenv('OPENAI_API_KEY'),
                                      model_name='text-embedding-3-small'
                                  ))

In [112]:
db.upsert(
    documents=L(all_egs).attrgot('text'),
    metadatas=[dict(label=l['label']) for l in all_egs],
    ids=[str(i) for i, _ in enumerate(all_egs)]
)

In [113]:
q="When can i expect to see the next episode of the show?"

In [86]:
rinspect(db.query, help=True)

╭───── <bound method Collection.query of Collection(id=1e72f6d3-c6ae-4f7e-a5f3-12e60deb982d, name=example)> ──────╮
│ def Collection.query(query_embeddings: Union[Sequence[float], Sequence[int], List[Union[Sequence[float],        │
│ Sequence[int]]], numpy.ndarray, List[numpy.ndarray], NoneType] = None, query_texts: Union[str, List[str],       │
│ NoneType] = None, query_images: Union[numpy.ndarray[Any, numpy.dtype[Union[numpy.uint64, numpy.int64,           │
│ numpy.float64]]], List[numpy.ndarray[Any, numpy.dtype[Union[numpy.uint64, numpy.int64, numpy.float64]]]],       │
│ NoneType] = None, query_uris: Union[str, List[str], NoneType] = None, n_results: int = 10, where:               │
│ Optional[Dict[Union[str, Literal['$and'], Literal['$or']], Union[str, int, float, bool,                         │
│ Dict[Union[Literal['$gt'], Literal['$gte'], Literal['$lt'], Literal['$lte'], Literal['$ne'], Literal['$eq'],    │
│ Literal['$and'], Literal['$or']], Union[str, int, float, bool]], Dict[Union[Literal['$in'], Literal['$nin']],   │
│ List[Union[str, int, float, bool]]], List[ForwardRef('Where')]]]] = None, where_document:                       │
│ Optional[Dict[Union[Literal['$contains'], Literal['$not_contains'], Literal['$and'], Literal['$or']],           │
│ Union[str, List[ForwardRef('WhereDocument')]]]] = None, include: List[chromadb.api.types.IncludeEnum] =         │
│ ['metadatas', 'documents', 'distances']) -> chromadb.api.types.QueryResult:                                     │
│                                                                                                                 │
│ Get the n_results nearest neighbor embeddings for provided query_embeddings or query_texts.                     │
│                                                                                                                 │
│ Args:                                                                                                           │
│     query_embeddings: The embeddings to get the closes neighbors of. Optional.                                  │
│     query_texts: The document texts to get the closes neighbors of. Optional.                                   │
│     query_images: The images to get the closes neighbors of. Optional.                                          │
│     n_results: The number of neighbors to return for each query_embedding or query_texts. Optional.             │
│     where: A Where type dict used to filter results by. E.g. `{"$and": [{"color" : "red"}, {"price": {"$gte":   │
│ 4.20}}]}`. Optional.                                                                                            │
│     where_document: A WhereDocument type dict used to filter by the documents. E.g. `{$contains: {"text":       │
│ "hello"}}`. Optional.                                                                                           │
│     include: A list of what to include in the results. Can contain `"embeddings"`, `"metadatas"`,               │
│ `"documents"`, `"distances"`. Ids are always included. Defaults to `["metadatas", "documents", "distances"]`.   │
│ Optional.                                                                                                       │
│                                                                                                                 │
│ Returns:                                                                                                        │
│     QueryResult: A QueryResult object containing the results.                                                   │
│                                                                                                                 │
│ Raises:                                                                                                         │
│     ValueError: If you don't provide either query_embeddings, query_texts, or query_images                      │
│     ValueError: If you provide both query_embeddings a

#### From MiniLM

In [87]:
res = db.query(query_texts=[q], n_results=5)

In [88]:
res

{'ids': [['9', '2', '8', '4', '6']],
 'distances': [[1.6789487600326538,
   1.7684942483901978,
   1.8034000396728516,
   1.8712923526763916,
   1.9162027835845947]],
 'metadatas': [[{'label': 'time_filter_requirement'},
   {'label': 'authorization'},
   {'label': 'time_filter_requirement'},
   {'label': 'drafting'},
   {'label': 'drafting'}]],
 'embeddings': None,
 'documents': [["Give me today's headlines from major news outlets.",
   "What's the weather like today?",
   'What are the current stock prices for Tech Co.?',
   'Write a summary of the quarterly report.',
   "What's the capital of France?"]],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [90]:
formatted_results = [
        (doc, metadata["label"], distance)
        for doc, metadata, distance in zip(
            res["documents"][0],
            res["metadatas"][0],
            res["distances"][0],
        )
    ]

In [91]:
formatted_results

[("Give me today's headlines from major news outlets.",
  'time_filter_requirement',
  1.6789487600326538),
 ("What's the weather like today?", 'authorization', 1.7684942483901978),
 ('What are the current stock prices for Tech Co.?',
  'time_filter_requirement',
  1.8034000396728516),
 ('Write a summary of the quarterly report.', 'drafting', 1.8712923526763916),
 ("What's the capital of France?", 'drafting', 1.9162027835845947)]

#### Using OpenAI

In [115]:
res = db.query(query_texts=[q], n_results=5)

In [116]:
res

{'ids': [['0', '14', '4', '8', '2']],
 'distances': [[1.6423436403274536,
   1.7078778743743896,
   1.7188336849212646,
   1.7208271026611328,
   1.722258448600769]],
 'metadatas': [[{'label': 'authorization'},
   {'label': 'error_symptoms'},
   {'label': 'drafting'},
   {'label': 'time_filter_requirement'},
   {'label': 'authorization'}]],
 'embeddings': None,
 'documents': [['Can you give me access to the financial reports?',
   'I can not view availables repositories to make an upgrade',
   'Write a summary of the quarterly report.',
   'What are the current stock prices for Tech Co.?',
   "What's the weather like today?"]],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [117]:
formatted_results = [
        (doc, metadata["label"], distance)
        for doc, metadata, distance in zip(
            res["documents"][0],
            res["metadatas"][0],
            res["distances"][0],
        )
    ]

In [118]:
formatted_results

[('Can you give me access to the financial reports?',
  'authorization',
  1.6423436403274536),
 ('I can not view availables repositories to make an upgrade',
  'error_symptoms',
  1.7078778743743896),
 ('Write a summary of the quarterly report.', 'drafting', 1.7188336849212646),
 ('What are the current stock prices for Tech Co.?',
  'time_filter_requirement',
  1.7208271026611328),
 ("What's the weather like today?", 'authorization', 1.722258448600769)]

In [119]:
template = Template(
            dedent(
                """
        Classify the following document:

        <doc>
        {{ query }}
        </doc>

        Similar examples:
        <examples>
        {% for doc, label, distance in formatted_results %}
        <example>
            <distance> {{ "%.2f"|format(distance) }} </distance>
            <label> {{ label }} </label>
            <similar_document> {{ doc }} </similar_document>
        </example>
        {% endfor %}
        </examples>

        Provide your classification based on the above information.
        """
            )
        )

In [122]:
print(template.render(query=q, formatted_results=formatted_results))


Classify the following document:

<doc>
When can i expect to see the next episode of the show?
</doc>

Similar examples:
<examples>

<example>
    <distance> 1.64 </distance>
    <label> authorization </label>
    <similar_document> Can you give me access to the financial reports? </similar_document>
</example>

<example>
    <distance> 1.71 </distance>
    <label> error_symptoms </label>
    <similar_document> I can not view availables repositories to make an upgrade </similar_document>
</example>

<example>
    <distance> 1.72 </distance>
    <label> drafting </label>
    <similar_document> Write a summary of the quarterly report. </similar_document>
</example>

<example>
    <distance> 1.72 </distance>
    <label> time_filter_requirement </label>
    <similar_document> What are the current stock prices for Tech Co.? </similar_document>
</example>

<example>
    <distance> 1.72 </distance>
    <label> authorization </label>
    <similar_document> What's the weather like today? </sim

I ran the above template against [Perplexity](https://www.perplexity.ai/search/classify-the-following-documen-3cI7GTs4RV2IV5ZXgMWZmw) and it worked great. 

Based on the given document and the similar examples provided, I would classify the document as:

Label: time_filter_requirement

Reasoning:
1. The query "When can i expect to see the next episode of the show?" is asking about a future time-based event, which aligns closely with the concept of a time filter requirement.

2. Among the provided examples, the closest match in terms of intent is the example labeled "time_filter_requirement" with the similar document "What are the current stock prices for Tech Co.?" Both this example and the given document are asking for current or upcoming information about a specific topic.

3. While the distances provided for the examples are all relatively close (ranging from 1.64 to 1.72), the "time_filter_requirement" label seems most appropriate for a question asking about the timing of a future event.

4. The other labels (authorization, error_symptoms, drafting) do not fit as well with the nature of the query, which is seeking information about a future occurrence rather than requesting access, reporting an error, or asking for content creation.

Therefore, considering the nature of the query and its similarity to the provided examples, classifying it as "time_filter_requirement" appears to be the most appropriate choice.